# Laboratorio 12: Pipeline de datos de la Semana de Ingeniería

Este notebook ejecuta las 8 etapas del pipeline paso a paso. La lógica de cada etapa vive en el paquete `pipeline/` del repositorio; aquí solo se llama y se muestran los resultados.

**Para correrlo:**
- **Local:** abre el notebook desde la carpeta `notebooks/` del repo.
- **Google Colab:** ejecuta primero la celda de Colab de abajo (clona el repo y te deja subir los archivos).

In [ ]:
# --- Solo en Google Colab (descomenta y cambia la URL por la de tu repo) ---
# !git clone https://github.com/<usuario>/pipeline-semana-ingenieria.git
# %cd pipeline-semana-ingenieria/notebooks

## Configuración

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from pipeline import (almacenamiento, cargar_datos, cargar_diccionario, limpieza,
                      seleccion, transformacion, validacion_de_informacion,
                      validar_carga, validar_layout)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
pd.set_option("display.max_columns", None)

# Rutas de entrada y salida (cámbialas por los archivos reales en data/raw/)
RUTA_DATOS = RAIZ / "data" / "raw" / "Programa_32IIT_21-09.xlsx"
RUTA_DICCIONARIO = RAIZ / "data" / "raw" / "Diccionario_Datos_Semana_Ingenieria.xlsx"
DIR_SALIDA = RAIZ / "output"

# Si no están los archivos reales, se usan los datos sintéticos de muestra
if not RUTA_DATOS.exists():
    RUTA_DATOS = RAIZ / "data" / "sample" / "programa_muestra.xlsx"
    RUTA_DICCIONARIO = RAIZ / "data" / "sample" / "diccionario_muestra.xlsx"
    print("Usando datos de muestra:", RUTA_DATOS.name)

# ETAPA 1 - CARGA
Lectura del programa de actividades y del diccionario de datos (layout y catálogos).

In [ ]:
df = cargar_datos(RUTA_DATOS)
diccionario = cargar_diccionario(RUTA_DICCIONARIO)

# ETAPA 2 - VALIDACIÓN DE CARGA
El archivo se leyó sin errores y no está vacío.

In [ ]:
carga_ok = validar_carga(df)
assert carga_ok, "No se puede continuar con el pipeline."
df.head()

# ETAPA 3 - VALIDACIÓN DE LAYOUT
Nombres de columnas, columnas requeridas y tipos de datos contra el diccionario.

In [ ]:
layout = validar_layout(df, diccionario)
print("Columnas faltantes:  ", layout.columnas_faltantes or "Ninguna")
print("Columnas adicionales:", layout.columnas_adicionales or "Ninguna")
print("Errores de tipo:     ", layout.errores_tipos or "Ninguno")
assert layout.estructura_valida, "No se puede continuar: faltan columnas requeridas."

# ETAPA 4 - VALIDACIÓN DE INFORMACIÓN
Número de registros que incumplen cada regla: duplicados, nulos y vacíos, formatos, rangos, catálogos y reglas entre campos.

In [ ]:
reporte_antes = validacion_de_informacion(df, diccionario)
reporte_antes

# ETAPA 5 - LIMPIEZA
Se corrigen los valores cuya corrección es determinística. La limpieza no elimina registros y deja cada cambio en una bitácora.

In [ ]:
df_limpio, bitacora = limpieza(df)
reporte_despues = validacion_de_informacion(df_limpio, diccionario)

reporte = reporte_antes.rename(columns={"Total Errores": "Antes de limpieza"})
reporte["Después de limpieza"] = reporte_despues["Total Errores"]
display(reporte)
bitacora

# ETAPA 6 - SELECCIÓN
Se conservan los registros que cumplen todas las reglas y los atributos necesarios. Los demás se separan con su motivo de rechazo.

In [ ]:
df_sel, df_rechazados = seleccion(df_limpio, diccionario, df_original=df)
df_rechazados

# ETAPA 7 - TRANSFORMACIÓN
Esquema de salida: nombres estándar, tipos de fecha y hora, atributos derivados, orden cronológico e identificador.

In [ ]:
df_final = transformacion(df_sel)
print(df_final.dtypes)
df_final.head()

# ETAPA 8 - ALMACENAMIENTO
Se guardan por separado los datos procesados y los registros rechazados, y se verifica la escritura releyendo cada archivo.

In [ ]:
rutas = almacenamiento(df_final, df_rechazados, DIR_SALIDA, df_bitacora=bitacora,
                       reportes={"validacion": reporte})
for nombre, ruta in rutas.items():
    print(f"{nombre:<11} -> {ruta.relative_to(RAIZ)}")